In [169]:
# ==============================================================================
# 🚫 CELDA BLOQUEADA: EVALUADOR "CAJA NEGRA" (CHECKPOINTS LOCALES)
# Ejecuta esta celda al iniciar y no modifiques su contenido.
# ==============================================================================
import pandas as pd
import numpy as np

class EvaluadorKaggleFYP:
    @staticmethod
    def auditar_estrella(df_fact, df_dim_creadores, df_dim_audios):
        print("🔍 Evaluando Checkpoint 1: Esquema Estrella...")
        if not df_dim_creadores['usr_key'].is_unique: return "❌ ERROR: 'usr_key' tiene duplicados."
        if not df_dim_audios['snd_key'].is_unique: return "❌ ERROR: 'snd_key' tiene duplicados."
        
        columnas_prohibidas = ['attr_tier', 'attr_tenure_m', 'attr_bpm']
        for col in columnas_prohibidas:
            if col in df_fact.columns: return f"❌ ERROR: La columna descriptiva '{col}' sigue en fact_telemetria."
                
        print("✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.\n")
        return True

    @staticmethod
    def auditar_cubo_olap(df_cubo):
        print("🔍 Evaluando Checkpoint 2: Cubo OLAP...")
        # En el set de entrenamiento hay ~700 videos orgánicos, en el de test hay ~300. 
        # Si la llave es única, el roll-up se hizo bien.
        if 'vid_key' not in df_cubo.columns or not df_cubo['vid_key'].is_unique:
            return "❌ ERROR: 'vid_key' no es único. No agrupaste a nivel de video."
            
        print("✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.\n")
        return True

evaluador = EvaluadorKaggleFYP()

# 📱 Hackeando el Algoritmo de TikTok (FYP)
Sigue estrictamente las reglas del **Diccionario de Telemetría** impreso. Recuerda que cualquier transformación que le apliques al set de Entrenamiento, deberás replicarla exactamente igual en el set de Prueba.

---

## Fase 1: Data Warehouse (Esquema Estrella)
**Objetivo:** Eliminar la redundancia.
Separa el archivo crudo (`train.csv`) en tres tablas (`df_dim_creadores`, `df_dim_audios`, y `df_fact_telemetria`) garantizando integridad referencial. Asegúrate de eliminar las columnas descriptivas de la tabla de hechos.

In [175]:
# NOTA: Si usas Kaggle directamente, la ruta de los archivos suele estar en '/kaggle/input/tu-competencia/'
df_train_raw = pd.read_csv('train.csv')
df_test_raw = pd.read_csv('test.csv')


# display(df_train_raw)
# display(df_test_raw)

# --- TU CÓDIGO AQUÍ: Construye el Esquema Estrella (usa df_train_raw para la auditoría local) ---
# DIMENSION CREADORES
df_dim_creadores = df_train_raw[['usr_key', 'attr_tier']].drop_duplicates().reset_index(drop=True)
df_dim_creadores['id_creador'] = df_dim_creadores.index + 1
df_dim_creadores = df_dim_creadores[['id_creador', 'usr_key', 'attr_tier']]

display(df_dim_creadores.head(5))

# DIMENSION AUDIO
df_dim_audios = df_train_raw[['snd_key']].drop_duplicates().reset_index(drop=True)
df_dim_audios['id_audio'] = df_dim_audios.index + 1
df_dim_audios = df_dim_audios[['id_audio', 'snd_key']]

display(df_dim_audios.head(5))

# TABLA DE HECHOS

df_temp = df_train_raw.merge(df_dim_creadores, on=['usr_key', 'attr_tier'], how='left')
df_temp = df_temp.merge(df_dim_audios, on=['snd_key'], how='left')

display(df_temp.head(5))

df_fact_telemetria = df_temp[['id_creador', 'id_audio', 'val_watch_sec', 'vid_key', 'val_swipe_pct', 'val_shares_cnt', 'sys_traffic_src', 'target_fyp_viral']]

display(df_fact_telemetria.head(5))

# Validación del Checkpoint 1
evaluador.auditar_estrella(df_fact_telemetria, df_dim_creadores, df_dim_audios)


,id_creador,usr_key,attr_tier
0,1,USR_0141,Top_Partner
1,2,USR_0216,Cuenta_Nueva
2,3,USR_0058,Cuenta_Nueva
3,4,USR_0051,Top_Partner
4,5,USR_0140,Top_Partner


,id_audio,snd_key
0,1,SND_0041
1,2,SND_0100
2,3,SND_0127
3,4,SND_0107
4,5,SND_0163


,log_event_id,vid_key,batch_hour_seq,sys_traffic_src,usr_key,attr_tier,attr_tenure_m,snd_key,attr_bpm,cat_code,pub_hour_utc,vid_len_sec,val_watch_sec,val_shares_cnt,val_swipe_pct,post_payout_usd,target_fyp_viral,id_creador,id_audio
0,EVT_101935,VID_00194,6,ORGANIC,USR_0141,Top_Partner,51,SND_0041,102,Storytime,11,15,5.30,20,0.635,0.00,0,1,1
1,EVT_106494,VID_00650,5,ORGANIC,USR_0216,Cuenta_Nueva,46,SND_0100,137,Storytime,5,15,-999.00,21,0.463,0.00,0,2,2
2,EVT_100360,VID_00037,1,ORGANIC,USR_0058,Cuenta_Nueva,28,SND_0127,154,Gaming,23,45,23.50,21,0.450,0.00,0,3,3
3,EVT_109663,VID_00967,4,ORGANIC,USR_0051,Top_Partner,46,SND_0041,102,Tech_Coding,14,30,11.37,14,0.597,0.00,0,4,1
4,EVT_108546,VID_00855,7,ORGANIC,USR_0140,Top_Partner,16,SND_0107,82,Storytime,16,45,29.08,16,0.374,166.73,1,5,4


,id_creador,id_audio,val_watch_sec,vid_key,val_swipe_pct,val_shares_cnt,sys_traffic_src,target_fyp_viral
0,1,1,5.30,VID_00194,0.635,20,ORGANIC,0
1,2,2,-999.00,VID_00650,0.463,21,ORGANIC,0
2,3,3,23.50,VID_00037,0.450,21,ORGANIC,0
3,4,1,11.37,VID_00967,0.597,14,ORGANIC,0
4,5,4,29.08,VID_00855,0.374,16,ORGANIC,1


🔍 Evaluando Checkpoint 1: Esquema Estrella...
✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.



True

## Fase 2: Operaciones OLAP (Cubo de Datos)
**Objetivo:** Limpieza y Granularidad.
1. Encuentra y repara los errores de los sensores de telemetría.
2. Filtra el tráfico ilegítimo (Bots).
3. Agrupa la telemetría transaccional para tener **1 sola fila por video**. Decide con inteligencia qué métricas se suman y cuáles se promedian.

In [176]:
# --- TU CÓDIGO AQUÍ:
df_cubo_train = df_fact_telemetria.merge(df_dim_creadores, on='id_creador').merge(df_dim_audios, on='id_audio')

df_fact_telemetria.loc[df_fact_telemetria['val_watch_sec'] < 0, 'val_watch_sec'] = df_fact_telemetria['val_watch_sec'].median()

df_fact_telemetria = df_fact_telemetria[df_fact_telemetria['sys_traffic_src'] == 'ORGANIC'].copy()

## val_watch_sec, val_swipe_pct, val_shares_cnt
# df_fact_telemetria_temp1 = df_fact_telemetria.groupby('vid_key')['val_watch_sec'].mean().reset_index()
# df_fact_telemetria_temp2 = df_fact_telemetria.groupby('vid_key')['val_swipe_pct'].mean().reset_index()
# df_fact_telemetria_temp3 = df_fact_telemetria.groupby('vid_key')['val_shares_cnt'].mean().reset_index()
#
# df_cubo_train = df_fact_telemetria_temp1.merge(df_fact_telemetria_temp1, on=['vid_key'], how='left').merge(df_fact_telemetria_temp2, on=['vid_key'], how='left').merge(df_fact_telemetria_temp3, on=['vid_key'], how='left')
#
# display(df_cubo_train)

agg_rules = {
    'val_watch_sec': 'mean',
    'val_swipe_pct': 'mean',
    'val_shares_cnt': 'sum',
    'id_creador': 'first',
    'id_audio': 'first',
    'target_fyp_viral': 'first',
}

df_cubo_train = df_fact_telemetria.groupby('vid_key', as_index=False).agg(agg_rules)

display(df_cubo_train.head(5))

# Validación del Checkpoint 2
evaluador.auditar_cubo_olap(df_cubo_train)

,vid_key,val_watch_sec,val_swipe_pct,val_shares_cnt,id_creador,id_audio,target_fyp_viral
0,VID_00001,19.942,0.3281,199,7,46,1
1,VID_00003,4.089,0.7393,122,89,123,0
2,VID_00004,7.948,0.4906,82,174,55,0
3,VID_00005,34.923,0.2243,156,154,122,1
4,VID_00006,13.762,0.7046,193,177,97,0


🔍 Evaluando Checkpoint 2: Cubo OLAP...
✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.



True

## Fase 3: Feature Engineering y Modelado
**Objetivo:** Traducir el negocio a matemáticas.
1. Une tus dimensiones de vuelta al cubo limpio.
2. Aplica codificación a las variables categóricas (cuidado con la memoria RAM, no uses One-Hot en columnas masivas).
3. Crea la métrica de negocio: **Hook Rate**.
4. Elimina IDs, variables originales transformadas y la trampa de *Data Leakage* antes de entrenar tu modelo.

In [181]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# --- TU CÓDIGO AQUÍ: Feature Engineering  ---

df_train = df_cubo_train.merge(df_dim_creadores, on='id_creador', how='left')
df_train = df_train.merge(df_dim_audios, on='id_audio', how='left')

df_train['hook_rate'] = 1.0 - X_train['val_swipe_pct']

y_train = df_train['target_fyp_viral'].astype(int)

X_train_raw = df_cubo_train.drop(['id_creador', 'id_audio', 'target_fyp_viral', 'vid_key'], axis=1)

display(X_train_raw.head(5))

X_test_limpio = X_train.copy()
# 1. Entrenar el Modelo
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
modelo.fit(X_test_limpio, y_train)

# 2. Predecir sobre el set de prueba ciego
predicciones = modelo.predict(X_test_limpio)

# 3. Exportar a Kaggle
submission = pd.DataFrame({
    'vid_key': X_test_limpio['vid_key'], # Asegúrate de que tu matriz final aún tenga el ID o guárdalo antes
    'target_fyp_viral': predicciones
})
submission.to_csv('submission.csv', index=False)
print("¡Archivo listo! Súbelo al Leaderboard.")

,val_watch_sec,val_swipe_pct,val_shares_cnt
0,19.942,0.3281,199
1,4.089,0.7393,122
2,7.948,0.4906,82
3,34.923,0.2243,156
4,13.762,0.7046,193


KeyError: 'vid_key'